In [1]:
from pathlib import Path

for name in ['perch_v2.onnx', 'sed_fold0.onnx', 'onnxruntime']:
    for p in Path('/kaggle/input').rglob(f'*{name}*'):
        print(f"{p.stat().st_size/1e6:.1f} MB  {p}")

409.1 MB  /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx
19.7 MB  /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold0.onnx
17.2 MB  /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
import subprocess, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

ONNX_WHL = "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", ONNX_WHL], check=True)

import onnxruntime as ort

BASE       = Path('/kaggle/input/competitions/birdclef-2026')
PERCH_PATH = "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx"

train    = pd.read_csv(BASE / 'train.csv')
taxonomy = pd.read_csv(BASE / 'taxonomy.csv')

def get_taxa_group(class_name):
    return {'Aves':0,'Insecta':1,'Amphibia':2,'Mammalia':3,'Reptilia':4}.get(class_name,-1)

train['taxa_group']    = train['class_name'].map(get_taxa_group)
taxonomy['taxa_group'] = taxonomy['class_name'].map(get_taxa_group)

all_species    = taxonomy['primary_label'].tolist()
species_to_idx = {s: i for i, s in enumerate(all_species)}

print(f"Train samples: {len(train)}")
print(f"Species:       {len(all_species)}")
print(f"Taxa groups:   {train['taxa_group'].value_counts().sort_index().to_dict()}")
print(f"GPU available: {torch.cuda.is_available()}")
print("Setup complete.")

Train samples: 35549
Species:       234
Taxa groups:   {0: 34799, 1: 199, 2: 451, 3: 99, 4: 1}
GPU available: True
Setup complete.


In [3]:
import re
from sklearn.model_selection import GroupKFold

soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')

SR          = 32000
N_WINDOWS   = 12
WINDOW_SAMPLES = SR * 5

PRIMARY_LABELS = taxonomy['primary_label'].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)

sc = (soundscape_labels
      .groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels)
      .reset_index(name="label_list"))

sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg","",regex=False) + "_" + sc["end_sec"].astype(str)
_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc    = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)

full_rows = (sc[sc["fully_labeled"]]
             .sort_values(["filename","end_sec"])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

print(f"Fully labeled files:  {len(full_files)}")
print(f"Total windows:        {len(full_rows)}")
print(f"Label matrix shape:   {Y_FULL.shape}")
print(f"Active classes:       {(Y_FULL.sum(0) > 0).sum()}")

# Taxa distribution in labels
taxa_in_labels = {}
for taxa_name, group_id in [('Aves',0),('Insecta',1),('Amphibia',2),('Mammalia',3),('Reptilia',4)]:
    species_in_group = taxonomy[taxonomy['taxa_group']==group_id]['primary_label'].tolist()
    indices = [label_to_idx[s] for s in species_in_group if s in label_to_idx]
    n_positive_windows = Y_FULL[:, indices].sum()
    taxa_in_labels[taxa_name] = n_positive_windows
    print(f"  {taxa_name:10s}: {n_positive_windows} positive windows in labeled data")

Fully labeled files:  59
Total windows:        708
Label matrix shape:   (708, 234)
Active classes:       71
  Aves      : 404 positive windows in labeled data
  Insecta   : 568 positive windows in labeled data
  Amphibia  : 2073 positive windows in labeled data
  Mammalia  : 29 positive windows in labeled data
  Reptilia  : 13 positive windows in labeled data


In [4]:
import soundfile as sf
import time

FILE_SAMPLES = 60 * SR

sess = ort.InferenceSession(PERCH_PATH, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
out_names  = {o.name: i for i, o in enumerate(sess.get_outputs())}

def extract_perch(paths, batch_size=8):
    all_emb    = []
    all_scores = []
    all_meta   = []

    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        x = np.zeros((len(batch)*N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)

        for bi, path in enumerate(batch):
            y, _ = sf.read(str(path), dtype='float32', always_2d=False)
            if y.ndim == 2: y = y.mean(1)
            if len(y) < FILE_SAMPLES: y = np.pad(y,(0,FILE_SAMPLES-len(y)))
            else: y = y[:FILE_SAMPLES]
            x[bi*N_WINDOWS:(bi+1)*N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_fname(Path(path).name)
            for w in range(N_WINDOWS):
                all_meta.append({
                    'filename': Path(path).name,
                    'site': meta['site'],
                    'hour_utc': meta['hour_utc'],
                    'window': w
                })

        outs   = sess.run(None, {input_name: x})
        emb    = outs[out_names['embedding']]
        labels = outs[out_names['label']]

        all_emb.append(emb)
        all_scores.append(labels)
        print(f"  Processed {min(i+batch_size, len(paths))}/{len(paths)} files")

    return (np.vstack(all_emb).astype(np.float32),
            np.vstack(all_scores).astype(np.float32),
            pd.DataFrame(all_meta))

print("Extracting Perch embeddings for 59 training soundscapes...")
t0 = time.time()

train_paths = [BASE / 'train_soundscapes' / fn for fn in full_files]
train_paths = [p for p in train_paths if p.exists()]

emb_tr, logits_tr, meta_tr = extract_perch(train_paths)

print(f"\nDone in {time.time()-t0:.1f}s")
print(f"Embeddings: {emb_tr.shape}")
print(f"Logits:     {logits_tr.shape}")

Extracting Perch embeddings for 59 training soundscapes...
  Processed 8/59 files
  Processed 16/59 files
  Processed 24/59 files
  Processed 32/59 files
  Processed 40/59 files
  Processed 48/59 files
  Processed 56/59 files
  Processed 59/59 files

Done in 115.1s
Embeddings: (708, 1536)
Logits:     (708, 14795)


In [5]:
# Build species-to-Perch-logit mapping from labels.csv
import pandas as pd
from pathlib import Path

LABELS_PATH = "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/labels.csv"
bc_labels = pd.read_csv(LABELS_PATH).reset_index()
bc_labels.columns = ['bc_index', 'scientific_name']

NO_LABEL = len(bc_labels)

mapping = taxonomy.merge(
    bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index('primary_label')['bc_index']

BC_INDICES  = [int(lbl2bc.loc[c]) if c in lbl2bc.index else NO_LABEL 
               for c in PRIMARY_LABELS]
MAPPED_MASK = [idx != NO_LABEL for idx in BC_INDICES]
MAPPED_POS  = [i for i, m in enumerate(MAPPED_MASK) if m]
MAPPED_BC   = [BC_INDICES[i] for i in MAPPED_POS]

print(f"Species with direct Perch logit: {sum(MAPPED_MASK)}/{N_CLASSES}")
# Map 14795-dim Perch logits → 234-dim competition logits (soundscapes)
perch_logits_sc = np.zeros((len(emb_tr), N_CLASSES), dtype=np.float32)
perch_logits_sc[:, MAPPED_POS] = logits_tr[:, MAPPED_BC]
print(f"Perch logits mapped for soundscapes: {perch_logits_sc.shape}")

Species with direct Perch logit: 203/234
Perch logits mapped for soundscapes: (708, 234)


In [6]:
# Extract Perch embeddings from training AUDIO files
# These 35,549 files give us species-level supervision

import soundfile as sf
from pathlib import Path

def extract_audio_embeddings(train_df, max_per_species=50, batch_size=32):
    """
    Extract Perch embeddings from training audio files.
    Cap at max_per_species per species to avoid bird domination.
    """
    all_emb    = []
    all_labels = []
    all_taxa   = []

    # Sample balanced across species
    sampled = (train_df.groupby('primary_label')
               .apply(lambda x: x.sample(min(len(x), max_per_species), 
                                          random_state=42))
               .reset_index(drop=True))

    print(f"Sampling {len(sampled)} audio files "
          f"({max_per_species} max per species)...")

    paths  = [BASE / 'train_audio' / fn for fn in sampled['filename']]
    labels = sampled['primary_label'].tolist()
    taxa   = sampled['taxa_group'].tolist()

    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        x = np.zeros((len(batch_paths), WINDOW_SAMPLES), dtype=np.float32)

        for bi, path in enumerate(batch_paths):
            try:
                y, sr = sf.read(str(path), dtype='float32', always_2d=False)
                if y.ndim == 2: y = y.mean(1)
                # Take middle 5 seconds
                if len(y) >= WINDOW_SAMPLES:
                    start = (len(y) - WINDOW_SAMPLES) // 2
                    y = y[start:start+WINDOW_SAMPLES]
                else:
                    y = np.pad(y, (0, WINDOW_SAMPLES-len(y)))
                x[bi] = y
            except:
                pass  # keep zeros if file fails

        outs = sess.run(None, {input_name: x})
        all_emb.append(outs[out_names['embedding']])
        all_labels.extend(labels[i:i+batch_size])
        all_taxa.extend(taxa[i:i+batch_size])

        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size, len(paths))}/{len(paths)} files")

    emb_arr  = np.vstack(all_emb).astype(np.float32)
    return emb_arr, all_labels, all_taxa

print("Extracting audio file embeddings...")
import time
t0 = time.time()
audio_emb, audio_labels, audio_taxa = extract_audio_embeddings(
    train, max_per_species=30, batch_size=32)
print(f"Done in {time.time()-t0:.1f}s")
print(f"Audio embeddings: {audio_emb.shape}")

# Build one-hot labels for audio files
audio_Y = np.zeros((len(audio_emb), N_CLASSES), dtype=np.float32)
for i, lbl in enumerate(audio_labels):
    if lbl in label_to_idx:
        audio_Y[i, label_to_idx[lbl]] = 1.0

# Extract REAL Perch logits for audio files
print("\nExtracting Perch logits for audio files...")
audio_logits_raw = []
audio_file_paths = [BASE / 'train_audio' / fn for fn in 
                    train[train['primary_label'].isin(audio_labels)]['filename'].tolist()[:len(audio_emb)]]

# Re-read audio and get logits (batch by batch)
sampled = (train.groupby('primary_label')
           .apply(lambda x: x.sample(min(len(x), 30), random_state=42), include_groups=False)
           .reset_index(drop=True))
paths_ordered = [BASE / 'train_audio' / fn for fn in sampled['filename']]

for i in range(0, len(paths_ordered), 32):
    batch_paths = paths_ordered[i:i+32]
    x = np.zeros((len(batch_paths), WINDOW_SAMPLES), dtype=np.float32)
    for bi, path in enumerate(batch_paths):
        try:
            y, _ = sf.read(str(path), dtype='float32', always_2d=False)
            if y.ndim == 2: y = y.mean(1)
            if len(y) >= WINDOW_SAMPLES:
                start = (len(y) - WINDOW_SAMPLES) // 2
                y = y[start:start+WINDOW_SAMPLES]
            else:
                y = np.pad(y, (0, WINDOW_SAMPLES - len(y)))
            x[bi] = y
        except: pass
    outs = sess.run(None, {input_name: x})
    audio_logits_raw.append(outs[out_names['label']])

audio_logits_full = np.vstack(audio_logits_raw).astype(np.float32)
perch_logits_audio = np.zeros((len(audio_emb), N_CLASSES), dtype=np.float32)
perch_logits_audio[:, MAPPED_POS] = audio_logits_full[:len(audio_emb), MAPPED_BC]

print(f"Audio Perch logits: {perch_logits_audio.shape}")
print(f"Combined dataset:")
print(f"  Soundscape windows: {len(emb_tr)}")
print(f"  Audio files:        {len(audio_emb)}")
print(f"  Total:              {len(emb_tr) + len(audio_emb)}")

Extracting audio file embeddings...
Sampling 5313 audio files (30 max per species)...


/tmp/ipykernel_23/629727337.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), max_per_species),


  32/5313 files
  672/5313 files
  1312/5313 files
  1952/5313 files
  2592/5313 files
  3232/5313 files
  3872/5313 files
  4512/5313 files
  5152/5313 files
Done in 1092.3s
Audio embeddings: (5313, 1536)

Extracting Perch logits for audio files...
Audio Perch logits: (5313, 234)
Combined dataset:
  Soundscape windows: 708
  Audio files:        5313
  Total:              6021


In [7]:
# Combine soundscape + audio embeddings
# Combine embeddings + Perch logits (1536 + 234 = 1770 dims)
emb_combined = np.vstack([
    np.hstack([emb_tr,    perch_logits_sc]),    # (708, 1770)
    np.hstack([audio_emb, perch_logits_audio])  # (5313, 1770)
]).astype(np.float32)

print(f"Combined input dim: {emb_combined.shape[1]}  ← should be 1770")
# Combined labels
audio_Y_combined = np.zeros((len(audio_emb), N_CLASSES), dtype=np.float32)
for i, lbl in enumerate(audio_labels):
    if lbl in label_to_idx:
        audio_Y_combined[i, label_to_idx[lbl]] = 1.0

Y_combined = np.vstack([Y_FULL, audio_Y_combined]).astype(np.float32)

print(f"Combined embeddings: {emb_combined.shape}")
print(f"Combined labels:     {Y_combined.shape}")
print(f"Active classes:      {(Y_combined.sum(0) > 0).sum()}")

# Taxa distribution check
taxa_combined = np.array(
    [train[train['primary_label']==lbl]['taxa_group'].values[0] 
     if lbl in train['primary_label'].values else 0 
     for lbl in audio_labels])

print("\nCombined taxa distribution:")
for name, gid in [('Aves',0),('Insecta',1),('Amphibia',2),
                   ('Mammalia',3),('Reptilia',4)]:
    # From soundscapes
    sc_idx = [label_to_idx[s] for s in 
              taxonomy[taxonomy['taxa_group']==gid]['primary_label'].tolist()
              if s in label_to_idx]
    sc_pos = Y_FULL[:, sc_idx].sum() if sc_idx else 0
    # From audio
    au_pos = (np.array(audio_taxa) == gid).sum()
    print(f"  {name:10s}: {int(sc_pos):4d} soundscape + {au_pos:4d} audio = {int(sc_pos)+au_pos}")

Combined input dim: 1770  ← should be 1770
Combined embeddings: (6021, 1770)
Combined labels:     (6021, 234)
Active classes:      234

Combined taxa distribution:
  Aves      :  404 soundscape + 4794 audio = 5198
  Insecta   :  568 soundscape +   48 audio = 616
  Amphibia  : 2073 soundscape +  384 audio = 2457
  Mammalia  :   29 soundscape +   86 audio = 115
  Reptilia  :   13 soundscape +    1 audio = 14


In [8]:
class CrossTaxaContrastiveAdapter(nn.Module):
    """
    Rectifies Perch embedding geometry across taxa.
    
    Perch was trained on birds → embedding space is organized
    around bird acoustics. Frog/insect calls land in distorted
    regions. This adapter learns a geometry that respects
    taxonomic boundaries.
    
    Novel element: taxonomic distance as contrastive metric.
    Same species → pull together
    Same taxa, diff species → mild pull
    Different taxa → push apart aggressively
    """
    def __init__(self, input_dim=1770, output_dim=512, hidden_dim=768, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.LayerNorm(output_dim)
        )

    def forward(self, x):
        return F.normalize(self.encoder(x), dim=-1)


def taxonomic_contrastive_loss(z, species_labels, taxa_labels,
                                temperature=0.07,
                                same_taxa_weight=0.3):
    """
    Novel loss: uses taxonomic distance as similarity metric.
    
    For each pair (i, j):
    - same species     → similarity target = 1.0  (pull together)
    - same taxa        → similarity target = 0.3  (mild attraction)
    - different taxa   → similarity target = 0.0  (push apart)
    
    This forces the adapter to learn a space that respects
    biological taxonomy, not just acoustic similarity.
    """
    n = z.shape[0]
    
    # Pairwise cosine similarity matrix
    sim = torch.matmul(z, z.T) / temperature  # (n, n)
    
    # Build target similarity matrix
    sp  = species_labels.unsqueeze(1)   # (n,1)
    tx  = taxa_labels.unsqueeze(1)      # (n,1)
    
    same_species = (sp == sp.T).float()
    same_taxa    = (tx == tx.T).float()
    diff_taxa    = 1.0 - same_taxa
    
    # Target: same species=1, same taxa=0.3, diff taxa=0
    target = (same_species +
               same_taxa_weight * same_taxa * (1 - same_species))
    target = target / (target.sum(dim=1, keepdim=True) + 1e-8)
    
    # Remove diagonal
    mask = ~torch.eye(n, dtype=torch.bool, device=z.device)
    sim  = sim[mask].reshape(n, n-1)
    target = target[mask].reshape(n, n-1)
    
    # Cross-entropy between predicted and target similarity
    log_softmax = F.log_softmax(sim, dim=1)
    loss = -(target * log_softmax).sum(dim=1).mean()
    
    return loss


# Test the adapter
adapter = CrossTaxaContrastiveAdapter().cuda()
dummy_emb = torch.randn(16, 1770).cuda()   
dummy_out    = adapter(dummy_emb)
dummy_sp     = torch.randint(0, 234, (16,)).cuda()
dummy_tx     = torch.randint(0, 5,   (16,)).cuda()
loss         = taxonomic_contrastive_loss(dummy_out, dummy_sp, dummy_tx)

print(f"Adapter output shape: {dummy_out.shape}")
print(f"Contrastive loss test: {loss.item():.4f}")
print(f"Adapter parameters: {sum(p.numel() for p in adapter.parameters()):,}")

Adapter output shape: torch.Size([16, 512])
Contrastive loss test: 1.2332
Adapter parameters: 2,348,544


In [9]:
class AvesHead(nn.Module):
    """Softmax head for birds — abundant data, standard classification."""
    def __init__(self, input_dim=512, n_species=182, dropout=0.2):  # was (self, taxonomy_df, input_dim=1770)
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_species)
        )
    def forward(self, x): return self.net(x)


class InsectaHead(nn.Module):
    """Softmax head for insects — moderate data, focal loss during training."""
    def __init__(self, input_dim=512, n_species=28, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_species)
        )
    def forward(self, x): return self.net(x)


class PrototypicalHead(nn.Module):
    """
    Prototypical head for scarce taxa (Amphibia, Mammalia).
    Classification = cosine similarity to learned class prototypes.
    Works with as few as 13 positive windows per species.
    """
    def __init__(self, input_dim=512, n_species=35, dropout=0.2):
        super().__init__()
        self.proj       = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.prototypes = nn.Parameter(torch.randn(n_species, 256))
        self.temperature = nn.Parameter(torch.tensor(10.0))
        nn.init.xavier_normal_(self.prototypes)

    def forward(self, x):
        h = F.normalize(self.proj(x), dim=-1)
        p = F.normalize(self.prototypes, dim=-1)
        return torch.matmul(h, p.T) * F.softplus(self.temperature)

    def init_prototypes_from_embeddings(self, embeddings, labels):
        """Initialize prototypes as mean of positive examples per class."""
        with torch.no_grad():
            h = F.normalize(self.proj(embeddings), dim=-1)
            for c in range(self.prototypes.shape[0]):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = h[mask].mean(0)


class TaxaMoE(nn.Module):
    """
    Taxa-conditional Mixture of Experts for PAM.
    
    Novel contributions:
    1. Cross-taxa contrastive adapter rectifies Perch geometry
    2. Learned taxa router for acoustic-based routing
    3. Specialist heads per taxa with data-appropriate losses
    4. Soft routing (not hard argmax) for robustness
    """
    def __init__(self, taxonomy_df, input_dim=1536):
        super().__init__()

        # Species per taxa group
        self.n_aves     = (taxonomy_df['taxa_group']==0).sum()
        self.n_insecta  = (taxonomy_df['taxa_group']==1).sum()
        self.n_amphibia = (taxonomy_df['taxa_group']==2).sum()
        self.n_mammalia = (taxonomy_df['taxa_group']==3).sum()
        self.n_reptilia = (taxonomy_df['taxa_group']==4).sum()
        self.n_classes  = len(taxonomy_df)

        # Species indices per taxa (for output assembly)
        self.register_buffer('aves_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==0].index.tolist()))
        self.register_buffer('insecta_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==1].index.tolist()))
        self.register_buffer('amphibia_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==2].index.tolist()))
        self.register_buffer('mammalia_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==3].index.tolist()))
        self.register_buffer('reptilia_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==4].index.tolist()))

        # Core modules
        self.adapter = CrossTaxaContrastiveAdapter(input_dim, 512)

        # Taxa router (5-way soft classifier)
        self.router = nn.Sequential(
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Linear(128, 5)
        )

        # Specialist heads
        self.aves_head     = AvesHead(512, self.n_aves)
        self.insecta_head  = InsectaHead(512, self.n_insecta)
        self.amphibia_head = PrototypicalHead(512, self.n_amphibia)
        self.mammalia_head = PrototypicalHead(512, self.n_mammalia)
        self.reptilia_head = PrototypicalHead(512, self.n_reptilia)

    def forward(self, x, return_router=False):
        # x: (batch, 1536)
        z = self.adapter(x)                          # (batch, 512)
        router_logits = self.router(z)               # (batch, 5)
        router_weights = F.softmax(router_logits, dim=-1)  # soft routing

        # Specialist predictions
        out_aves     = self.aves_head(z)
        out_insecta  = self.insecta_head(z)
        out_amphibia = self.amphibia_head(z)
        out_mammalia = self.mammalia_head(z)
        out_reptilia = self.reptilia_head(z)

        # Assemble full 234-class output
        batch = x.shape[0]
        logits = torch.zeros(batch, self.n_classes, device=x.device)

        logits[:, self.aves_idx]     = out_aves     * router_weights[:, 0:1]
        logits[:, self.insecta_idx]  = out_insecta  * router_weights[:, 1:2]
        logits[:, self.amphibia_idx] = out_amphibia * router_weights[:, 2:3]
        logits[:, self.mammalia_idx] = out_mammalia * router_weights[:, 3:4]
        logits[:, self.reptilia_idx] = out_reptilia * router_weights[:, 4:5]

        if return_router:
            return logits, router_weights
        return logits

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Reset taxonomy index for correct species ordering
taxonomy_indexed = taxonomy.reset_index(drop=True)

model = TaxaMoE(taxonomy_indexed, input_dim=emb_tr.shape[1]).cuda()
print(f"TaxaMoE parameters: {model.count_parameters():,}")

# Test forward pass
dummy = torch.randn(32, 1536).cuda()
out, router_w = model(dummy, return_router=True)
print(f"Output shape:        {out.shape}")
print(f"Router weights mean: {router_w.mean(0).detach().cpu().numpy().round(3)}")
print("TaxaMoE architecture ready.")

TaxaMoE parameters: 2,884,934
Output shape:        torch.Size([32, 234])
Router weights mean: [0.209 0.189 0.215 0.197 0.19 ]
TaxaMoE architecture ready.


In [10]:
from torch.utils.data import Dataset, DataLoader

class SoundscapeDataset(Dataset):
    def __init__(self, embeddings, labels, taxa_labels, species_labels):
        self.embeddings    = torch.tensor(embeddings, dtype=torch.float32)
        self.labels        = torch.tensor(labels,     dtype=torch.float32)
        self.taxa_labels   = torch.tensor(taxa_labels,  dtype=torch.long)
        self.species_labels = torch.tensor(species_labels, dtype=torch.long)

    def __len__(self): return len(self.embeddings)
    def __getitem__(self, i):
        return self.embeddings[i], self.labels[i], self.taxa_labels[i], self.species_labels[i]


def focal_loss(logits, targets, gamma=2.0, pos_weight=None):
    bce   = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction='none')
    probs = torch.sigmoid(logits)
    pt    = torch.where(targets > 0.5, probs, 1 - probs)
    return (((1 - pt) ** gamma) * bce).mean()


def train_taxamoe(emb_tr, Y_full, taxonomy_df,
                  n_epochs=60, lr=3e-4, batch_size=64,
                  contrastive_weight=0.3):

    # Build per-sample taxa and species labels
    primary_labels = taxonomy_df['primary_label'].tolist()
    taxa_per_sample = np.zeros(len(emb_tr), dtype=np.int64)
    sp_per_sample   = np.zeros(len(emb_tr), dtype=np.int64)

    for i in range(len(emb_tr)):
        active = np.where(Y_full[i] > 0)[0]
        if len(active) > 0:
            sp_idx           = active[0]
            taxa_per_sample[i] = taxonomy_df.iloc[sp_idx]['taxa_group']
            sp_per_sample[i]   = sp_idx
        else:
            taxa_per_sample[i] = 0
            sp_per_sample[i]   = 0

    dataset = SoundscapeDataset(emb_tr, Y_full, taxa_per_sample, sp_per_sample)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    # Reset taxonomy index
    taxonomy_indexed = taxonomy_df.reset_index(drop=True)
    model = TaxaMoE(taxonomy_indexed, input_dim=emb_tr.shape[1]).cuda()

    # Pos weights for imbalanced classes
    pos_counts  = torch.tensor(Y_full.sum(0) + 1, dtype=torch.float32).cuda()
    total       = Y_full.shape[0]
    pos_weights = ((total - pos_counts) / pos_counts).clamp(max=25.0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        epochs=n_epochs, steps_per_epoch=len(loader),
        pct_start=0.1, anneal_strategy='cos')

    best_loss, best_state = float('inf'), None

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_cls  = 0.0
        epoch_ctr  = 0.0

        for emb_b, lab_b, taxa_b, sp_b in loader:
            emb_b  = emb_b.cuda()
            lab_b  = lab_b.cuda()
            taxa_b = taxa_b.cuda()
            sp_b   = sp_b.cuda()

            logits, router_w = model(emb_b, return_router=True)

            # Classification loss (focal for all taxa)
            cls_loss = focal_loss(logits, lab_b,
                                   gamma=2.0, pos_weight=pos_weights)

            # Contrastive loss on adapter output
            z = model.adapter(emb_b)
            ctr_loss = taxonomic_contrastive_loss(z, sp_b, taxa_b)

            # Router entropy loss (encourage decisive routing)
            router_entropy = -(router_w * (router_w + 1e-8).log()).sum(dim=1).mean()

            loss = cls_loss + contrastive_weight * ctr_loss + 0.01 * router_entropy

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            epoch_cls  += cls_loss.item()
            epoch_ctr  += ctr_loss.item()

        avg_loss = epoch_loss / len(loader)
        if avg_loss < best_loss:
            best_loss  = avg_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d} | loss={avg_loss:.4f} "
                  f"cls={epoch_cls/len(loader):.4f} "
                  f"ctr={epoch_ctr/len(loader):.4f}")

    model.load_state_dict(best_state)
    print(f"\nBest loss: {best_loss:.4f}")
    return model


print("Training TaxaMoE...")
import time
t0 = time.time()
taxamoe_model = train_taxamoe(emb_tr, Y_FULL, taxonomy, n_epochs=60)
print(f"Training time: {time.time()-t0:.1f}s")

Training TaxaMoE...
Epoch  10 | loss=1.0396 cls=0.0686 ctr=3.2029
Epoch  20 | loss=0.9944 cls=0.0406 ctr=3.1420
Epoch  30 | loss=0.9808 cls=0.0298 ctr=3.1331
Epoch  40 | loss=0.9748 cls=0.0250 ctr=3.1294
Epoch  50 | loss=0.9738 cls=0.0234 ctr=3.1312
Epoch  60 | loss=0.9710 cls=0.0233 ctr=3.1223

Best loss: 0.9696
Training time: 16.4s


In [11]:
from sklearn.metrics import roc_auc_score

def evaluate_taxamoe(model, embeddings, labels, batch_size=128):
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for i in range(0, len(embeddings), batch_size):
            batch = torch.tensor(
                embeddings[i:i+batch_size], dtype=torch.float32).cuda()
            logits = model(batch)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
    
    preds = np.vstack(all_preds)
    
    # Competition metric: macro AUC skipping empty classes
    keep = labels.sum(0) > 0
    auc  = roc_auc_score(labels[:, keep], preds[:, keep], average='macro')
    
    # Per-taxa AUC
    for taxa_name, group_id in [('Aves',0),('Insecta',1),
                                  ('Amphibia',2),('Mammalia',3),('Reptilia',4)]:
        sp_idx = taxonomy[taxonomy['taxa_group']==group_id].index.tolist()
        sp_idx = [i for i in sp_idx if labels[:, i].sum() > 0]
        if len(sp_idx) > 0:
            taxa_auc = roc_auc_score(
                labels[:, sp_idx], preds[:, sp_idx], average='macro')
            print(f"  {taxa_name:10s} AUC: {taxa_auc:.4f} ({len(sp_idx)} active species)")
    
    return auc, preds

print("=== TAXAMOE EVALUATION (train set — upper bound) ===")
auc, preds = evaluate_taxamoe(taxamoe_model, emb_tr, Y_FULL)
print(f"\nOverall macro AUC: {auc:.4f}")

# Router analysis — what is the model actually routing?
print("\n=== ROUTER ANALYSIS ===")
taxamoe_model.eval()
with torch.no_grad():
    sample = torch.tensor(emb_tr[:100], dtype=torch.float32).cuda()
    _, router_w = taxamoe_model(sample, return_router=True)
    rw = router_w.cpu().numpy()

taxa_names = ['Aves', 'Insecta', 'Amphibia', 'Mammalia', 'Reptilia']
print("Mean routing weights after training:")
for i, name in enumerate(taxa_names):
    print(f"  {name:10s}: {rw[:, i].mean():.3f}")

# Save model
torch.save({
    'model_state': taxamoe_model.state_dict(),
    'taxonomy':    taxonomy.to_dict(),
    'auc':         auc,
}, '/kaggle/working/taxamoe_model.pt')
print("\nModel saved to /kaggle/working/taxamoe_model.pt")

=== TAXAMOE EVALUATION (train set — upper bound) ===
  Aves       AUC: 0.9963 (25 active species)
  Insecta    AUC: 0.9992 (25 active species)
  Amphibia   AUC: 0.9928 (17 active species)
  Mammalia   AUC: 0.9318 (3 active species)
  Reptilia   AUC: 1.0000 (1 active species)

Overall macro AUC: 0.9938

=== ROUTER ANALYSIS ===
Mean routing weights after training:
  Aves      : 0.412
  Insecta   : 0.310
  Amphibia  : 0.259
  Mammalia  : 0.011
  Reptilia  : 0.008

Model saved to /kaggle/working/taxamoe_model.pt


In [12]:
print("Retraining TaxaMoE on combined dataset...")
t0 = time.time()

taxamoe_model_v3 = train_taxamoe(
    emb_combined, Y_combined, taxonomy,
    n_epochs=300,          # was 120
    lr=1e-3,               # higher peak LR for OneCycle
    batch_size=256,        # larger batch = better contrastive pairs
    contrastive_weight=0.15)  # reduce contrastive weight, focus on classification

print(f"Training time: {time.time()-t0:.1f}s")

# Evaluate
print("\n=== V3 EVALUATION ===")
auc_v3, preds_v3 = evaluate_taxamoe(taxamoe_model_v3, emb_combined, Y_combined)
print(f"\nOverall macro AUC: {auc_v3:.4f}")

# Save v2
torch.save({
    'model_state': taxamoe_model_v3.state_dict(),
    'taxonomy':    taxonomy.to_dict(),
    'auc':         auc_v3,
    'version':     'v2_combined'
}, '/kaggle/working/taxamoe_model_v3.pt')
print("Saved: taxamoe_model_v3.pt")

Retraining TaxaMoE on combined dataset...
Epoch  10 | loss=0.8375 cls=0.0961 ctr=4.8566
Epoch  20 | loss=0.7594 cls=0.0336 ctr=4.7795
Epoch  30 | loss=0.7413 cls=0.0183 ctr=4.7635
Epoch  40 | loss=0.7350 cls=0.0140 ctr=4.7521
Epoch  50 | loss=0.7319 cls=0.0118 ctr=4.7482
Epoch  60 | loss=0.7308 cls=0.0110 ctr=4.7480
Epoch  70 | loss=0.7385 cls=0.0148 ctr=4.7742
Epoch  80 | loss=0.7295 cls=0.0100 ctr=4.7488
Epoch  90 | loss=0.7285 cls=0.0094 ctr=4.7476
Epoch 100 | loss=0.7282 cls=0.0096 ctr=4.7450
Epoch 110 | loss=0.7273 cls=0.0088 ctr=4.7459
Epoch 120 | loss=0.7267 cls=0.0088 ctr=4.7423
Epoch 130 | loss=0.7268 cls=0.0084 ctr=4.7469
Epoch 140 | loss=0.7263 cls=0.0084 ctr=4.7448
Epoch 150 | loss=0.7261 cls=0.0087 ctr=4.7419
Epoch 160 | loss=0.7258 cls=0.0081 ctr=4.7444
Epoch 170 | loss=0.7263 cls=0.0088 ctr=4.7432
Epoch 180 | loss=0.7253 cls=0.0080 ctr=4.7423
Epoch 190 | loss=0.7254 cls=0.0079 ctr=4.7446
Epoch 200 | loss=0.7248 cls=0.0079 ctr=4.7408
Epoch 210 | loss=0.7250 cls=0.0079 ctr

In [13]:
def local_cv_score(model, embeddings, labels, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros_like(labels, dtype=np.float32)
    
    # Move model to CPU for evaluation
    model = model.cpu()
    model.eval()
    
    for fold, (tr_idx, va_idx) in enumerate(
            gkf.split(embeddings, groups=groups), 1):
        with torch.no_grad():
            batch  = torch.tensor(embeddings[va_idx], dtype=torch.float32)
            logits = model(batch)
            probs  = torch.sigmoid(logits).numpy()
        oof[va_idx] = probs
        
        keep = labels[va_idx].sum(0) > 0
        if keep.sum() > 0:
            fold_auc = roc_auc_score(
                labels[va_idx][:, keep],
                probs[:, keep],
                average='macro')
            print(f"  Fold {fold}: {fold_auc:.4f}")
    
    keep    = labels.sum(0) > 0
    oof_auc = roc_auc_score(
        labels[:, keep], oof[:, keep], average='macro')
    print(f"\nOOF AUC: {oof_auc:.4f} ← estimate of LB score")
    return oof_auc

groups = meta_tr['filename'].values
sc_groups       = meta_tr['filename'].values
audio_groups    = np.array([f"audio_{lbl}" for lbl in audio_labels])
combined_groups = np.concatenate([sc_groups, audio_groups])

print("=== LOCAL CV SCORE ===")
local_cv_score(taxamoe_model_v3, emb_combined, Y_combined, combined_groups)

=== LOCAL CV SCORE ===
  Fold 1: 1.0000
  Fold 2: 0.9882
  Fold 3: 0.9876
  Fold 4: 1.0000
  Fold 5: 0.9886

OOF AUC: 0.9988 ← estimate of LB score


np.float64(0.9987503299269632)

In [14]:
# ── Train BiSSM LightProtoSSM ─────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
], check=True)
import onnxruntime as ort

class BiSSMLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.fwd  = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)
        self.bwd  = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        xf = F.gelu(self.fwd(x.transpose(1,2)).transpose(1,2))
        xb = F.gelu(self.bwd(x.flip(1).transpose(1,2)).transpose(1,2).flip(1))
        return self.norm(x + xf + xb)

class BiSSMLightProto(nn.Module):
    def __init__(self, n_classes=234, embed_dim=1536, d_model=128, dropout=0.5):
        super().__init__()
        self.proj     = nn.Linear(embed_dim, d_model)
        self.ssm1     = BiSSMLayer(d_model)
        self.ssm2     = BiSSMLayer(d_model)
        self.attn     = nn.MultiheadAttention(d_model, num_heads=2, batch_first=True)
        self.norm     = nn.LayerNorm(d_model)
        self.head     = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, n_classes))
    def forward(self, x):
        h = F.gelu(self.proj(x))
        h = self.ssm1(h)
        h = self.ssm2(h)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a)
        return self.head(h)

# Use soundscape data reshaped to (n_files, 24, 1536)
n_sc   = len(full_files)
SC_WIN = emb_tr.shape[0] // n_sc   # windows per file (24)
emb_3d = emb_tr.reshape(n_sc, SC_WIN, 1536)
Y_3d   = Y_FULL.reshape(n_sc, SC_WIN, N_CLASSES)

# Convert to tensors ONCE before training
x = torch.tensor(emb_3d, dtype=torch.float32).cuda()
y = torch.tensor(Y_3d,   dtype=torch.float32).cuda()

pos_w = torch.tensor(
    ((len(Y_FULL) - Y_FULL.sum(0) + 1) / (Y_FULL.sum(0) + 1)),
    dtype=torch.float32).cuda().clamp(max=20.0)

# Train 3 seeds, average predictions
all_bissm_preds = []

for seed in [42, 7, 123]:
    torch.manual_seed(seed)
    model = BiSSMLightProto(N_CLASSES, dropout=0.50).cuda()   # .cuda() added
    opt   = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

    best_loss, best_state = float('inf'), None
    for epoch in range(50):
        model.train()
        # Add time-dropout augmentation: randomly zero 2-3 windows
        x_aug = x.clone()
        for _ in range(2):
            t = torch.randint(0, SC_WIN, (1,)).item()
            x_aug[:, t, :] *= 0.0
        logits = model(x_aug)
        loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_w)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch+1) % 20 == 0:
            print(f"  Seed {seed} Epoch {epoch+1}: {loss.item():.4f}")

    model.load_state_dict(best_state)
    all_bissm_preds.append(best_state)
    print(f"Seed {seed} best loss: {best_loss:.4f}")

# Save the 3-seed averaged model (save all 3 state dicts)
torch.save({
    'seed_states': all_bissm_preds,
    'n_classes':   N_CLASSES,
    'SC_WIN':      SC_WIN,
}, '/kaggle/working/bissm_proto.pt')

# Export seed 0 to ONNX for fast CPU inference
model.load_state_dict(all_bissm_preds[0])
model.cpu().eval()
dummy = torch.randn(1, SC_WIN, 1536)
torch.onnx.export(
    model, dummy, '/kaggle/working/bissm_proto.onnx',
    opset_version=14,
    input_names=['embeddings'], output_names=['logits'],
    dynamic_axes={'embeddings': {0:'batch'}, 'logits': {0:'batch'}},
    dynamo=False
)

# Verify
sess = ort.InferenceSession('/kaggle/working/bissm_proto.onnx',
                             providers=['CPUExecutionProvider'])
out  = sess.run(None, {'embeddings': dummy.numpy()})
print(f"\nONNX output: {out[0].shape}")
print(f"Model size:  {__import__('os').path.getsize('/kaggle/working/bissm_proto.onnx')/1e6:.1f} MB")
print("Done. Upload bissm_proto.onnx to your dataset.")

  Seed 42 Epoch 20: 0.4708
  Seed 42 Epoch 40: 0.3740
Seed 42 best loss: 0.3690
  Seed 7 Epoch 20: 0.4681
  Seed 7 Epoch 40: 0.3742
Seed 7 best loss: 0.3681
  Seed 123 Epoch 20: 0.4755
  Seed 123 Epoch 40: 0.3848
Seed 123 best loss: 0.3751


/tmp/ipykernel_23/669949570.py:93: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(



ONNX output: (1, 12, 234)
Model size:  1.2 MB
Done. Upload bissm_proto.onnx to your dataset.


In [15]:
# ── Train LightProtoSSM on 59 labeled soundscapes ────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
], check=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
import numpy as np

class LightProtoSSM(nn.Module):
    def __init__(self, n_classes=234, embed_dim=1536, hidden=320, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(embed_dim, hidden)
        self.gru  = nn.GRU(hidden, hidden, num_layers=2, batch_first=True,
                           bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden*2),
            nn.Dropout(dropout),
            nn.Linear(hidden*2, n_classes)
        )
    def forward(self, x):
        h = F.gelu(self.proj(x))
        h, _ = self.gru(h)
        return self.head(h)

n_sc_files = len(full_files)
emb_sc_3d  = emb_tr.reshape(n_sc_files, N_WINDOWS, 1536)
Y_sc_3d    = Y_FULL.reshape(n_sc_files, N_WINDOWS, N_CLASSES)

pos_counts  = torch.tensor(Y_FULL.sum(0) + 1, dtype=torch.float32).cuda()
pos_weights = ((len(Y_FULL) - pos_counts) / pos_counts).clamp(max=20.0)

proto_model = LightProtoSSM(N_CLASSES).cuda()
opt         = torch.optim.AdamW(proto_model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=150)

print("Training LightProtoSSM on 59 soundscapes...")
best_loss, best_state = float('inf'), None

for epoch in range(150):
    proto_model.train()
    x      = torch.tensor(emb_sc_3d, dtype=torch.float32).cuda()
    y      = torch.tensor(Y_sc_3d,   dtype=torch.float32).cuda()
    logits = proto_model(x)
    loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weights)
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(proto_model.parameters(), 1.0)
    opt.step()
    scheduler.step()
    if loss.item() < best_loss:
        best_loss  = loss.item()
        best_state = {k: v.clone() for k, v in proto_model.state_dict().items()}
    if (epoch+1) % 30 == 0:
        print(f"  Epoch {epoch+1}: loss={loss.item():.4f}")

proto_model.load_state_dict(best_state)
print(f"Best loss: {best_loss:.4f}")

proto_model.cpu().eval()
dummy = torch.randn(1, N_WINDOWS, 1536)

torch.onnx.export(
    proto_model, dummy,
    '/kaggle/working/lightprotossm.onnx',
    opset_version=14,
    input_names=['embeddings'],
    output_names=['logits'],
    dynamic_axes={'embeddings': {0: 'batch'}, 'logits': {0: 'batch'}},
    dynamo=False
)
print("Saved: lightprotossm.onnx")

sess_check = ort.InferenceSession('/kaggle/working/lightprotossm.onnx',
                                   providers=['CPUExecutionProvider'])
out_check  = sess_check.run(None, {'embeddings': dummy.numpy()})
print(f"ONNX output shape: {out_check[0].shape}")
print(f"Model size: {__import__('os').path.getsize('/kaggle/working/lightprotossm.onnx')/1e6:.1f} MB")

Training LightProtoSSM on 59 soundscapes...
  Epoch 30: loss=0.1098
  Epoch 60: loss=0.0587
  Epoch 90: loss=0.0447
  Epoch 120: loss=0.0406
  Epoch 150: loss=0.0405
Best loss: 0.0397


/tmp/ipykernel_23/4117464490.py:66: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4553: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


Saved: lightprotossm.onnx
ONNX output shape: (1, 12, 234)
Model size: 14.9 MB
